In [3]:
import torch

print("===== PyTorch CUDA Check =====")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for idx in range(torch.cuda.device_count()):
        print(f"GPU {idx}: {torch.cuda.get_device_name(idx)}")
    # test for GPU
    a = torch.randn(2048, 2048).cuda()
    b = a @ a
    print("GPU test passed！")
else:
    print("CUDA unavailable")


===== PyTorch CUDA Check =====
PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU 0: NVIDIA GeForce RTX 4050 Laptop GPU
GPU test passed！


In [1]:
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
import hashlib
import json
import platform
import struct
import time

import pandas as pd
import PIL
from PIL import Image
from IPython.display import display

In [14]:
PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "plantseg"
OUT_DIR = PROJECT_DIR / "artifacts" / "data_cleaning"

HOSTS = {
    "apple", "banana", "citrus", "corn", "potato",
    "soybean", "tomato", "wheat", "zucchini"
}

SPLITS = ["train", "val", "test"]

SPLIT_MAP = {
    "Training": "train",
    "Validation": "val",
    "Test": "test"
}


In [7]:
# ============================================================
# 2. 读取元数据，筛选九种植物，建立固定类别映射
# ============================================================

metadata_path = DATA_DIR / "Metadata.csv"

meta = pd.read_csv(
    metadata_path,
    dtype=str,
    keep_default_na=False
)

required = {"Name", "Index", "Plant", "Disease", "Split"}

if not required.issubset(meta.columns):
    raise ValueError(f"元数据缺少字段：{required - set(meta.columns)}")

audit_df = meta.loc[
    meta["Plant"].str.strip().str.lower().isin(HOSTS)
].copy()

if audit_df.empty:
    raise ValueError("没有找到指定九种植物，请检查数据目录和元数据。")

if not set(audit_df["Split"]).issubset(SPLIT_MAP):
    raise ValueError("发现未知的 Split 标签，请检查元数据。")

if any(
    Path(name).name != name or "\\" in name
    for name in audit_df["Name"]
):
    raise ValueError("Name 应为不含目录的图片文件名。")

audit_df["split"] = audit_df["Split"].map(SPLIT_MAP)

# 路径相对于 DATA_DIR；不把本地绝对路径写入实验清单。
audit_df["relative_path"] = (
    "images/" + audit_df["split"] + "/" + audit_df["Name"]
)

audit_df["record_id"] = audit_df["relative_path"]

if audit_df["record_id"].duplicated().any():
    raise ValueError(
        "元数据存在重复记录，请先核查，不能将其直接计为不同图片。"
    )

# 固定记录顺序，便于复现清单及其校验值。
audit_df = (
    audit_df.sort_values("record_id")
    .reset_index(drop=True)
)

# 在清洗前的完整子集中建立类别映射。
# 保留原始列，包括 Plant、Disease、Index、URL、License 等。
class_df = (
    audit_df[["Plant", "Disease"]]
    .drop_duplicates()
    .sort_values(["Plant", "Disease"])
    .reset_index(drop=True)
)

class_df.insert(0, "class_id", range(len(class_df)))

class_lookup = {
    (row.Plant, row.Disease): int(row.class_id)
    for row in class_df.itertuples(index=False)
}

audit_df["class_id"] = [
    class_lookup[(plant, disease)]
    for plant, disease in zip(
        audit_df["Plant"], audit_df["Disease"]
    )
]

In [8]:
# ============================================================
# 3. 检查图片并计算两种 SHA-256
# ============================================================

def inspect_image(relative_path):
    result = {
        "file_sha256": "",
        "rgb_sha256": "",
        "width": None,
        "height": None,
        "image_error": ""
    }

    path = DATA_DIR / relative_path

    try:
        raw = path.read_bytes()

        # 原始文件字节的 SHA-256。
        result["file_sha256"] = hashlib.sha256(raw).hexdigest()

        # 从同一份文件字节解码，保证两种哈希对应同一次读取。
        with Image.open(BytesIO(raw)) as image:
            image.load()
            rgb = image.convert("RGB")
            width, height = rgb.size

            # 固定 RGB 哈希定义：
            # 版本前缀 + 两个大端 uint64 宽高 + 按行排列的 RGB 字节。
            #
            # 不缩放、不裁剪、不做 EXIF 转向、不做任何增强。
            digest = hashlib.sha256(b"PlantSeg-RGB-v1\0")
            digest.update(struct.pack(">QQ", width, height))
            digest.update(rgb.tobytes())

            result.update(
                rgb_sha256=digest.hexdigest(),
                width=width,
                height=height
            )

    except Exception as exc:
        reason = (
            "missing_file"
            if not path.is_file()
            else "unreadable_image"
        )

        result["image_error"] = (
            f"{reason}: {type(exc).__name__}: {exc}"
        )

    return result


print(
    f"检查 {len(audit_df):,} 张图片、{len(class_df)} 类；"
    "正在解码并计算哈希……"
)

# 线程池适合这里的文件读取与图片解码；
# 不使用 Windows Notebook 中较麻烦的多进程。
with ThreadPoolExecutor(max_workers=4) as pool:
    checks = list(
        pool.map(inspect_image, audit_df["relative_path"])
    )

audit_df = pd.concat(
    [audit_df, pd.DataFrame(checks)],
    axis=1
)

for column in ["width", "height"]:
    audit_df[column] = audit_df[column].astype("Int64")

检查 4,349 张图片、43 类；正在解码并计算哈希……


In [9]:
# ============================================================
# 4. 合并两种重复关系，所有重复组整组排除
# ============================================================

# 并查集用于合并关联记录：
# 文件哈希相同 OR RGB 哈希相同，都属于同一个连通组。
parent = list(range(len(audit_df)))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def union(a, b):
    a, b = find(a), find(b)
    if a != b:
        parent[b] = a


# 无法读取的记录单独排除，不参与正常图像的重复分组。
valid = audit_df["image_error"].eq("")

for column in ["file_sha256", "rgb_sha256"]:
    seen = {}

    for i in audit_df.index[valid]:
        value = audit_df.at[i, column]

        if value in seen:
            union(i, seen[value])
        else:
            seen[value] = i

groups = defaultdict(list)

for i in audit_df.index[valid]:
    groups[find(i)].append(i)

audit_df["duplicate_group_id"] = ""
audit_df["group_size"] = 0
audit_df["cross_split"] = False
audit_df["label_conflict"] = False
audit_df["included"] = valid
audit_df["exclusion_reason"] = audit_df["image_error"]

for members in groups.values():
    audit_df.loc[members, "group_size"] = len(members)

    # 唯一图像保留；重复组全部排除。
    if len(members) < 2:
        continue

    block = audit_df.loc[members]

    # 使用组内固定记录标识生成稳定的重复组 ID。
    group_id = "dup_" + hashlib.sha256(
        json.dumps(
            sorted(block["record_id"]),
            ensure_ascii=False
        ).encode("utf-8")
    ).hexdigest()

    audit_df.loc[members, "duplicate_group_id"] = group_id

    audit_df.loc[members, "cross_split"] = (
        block["split"].nunique() > 1
    )

    audit_df.loc[members, "label_conflict"] = (
        block["class_id"].nunique() > 1
    )

    audit_df.loc[members, "included"] = False

    audit_df.loc[members, "exclusion_reason"] = (
        "exact_duplicate_group"
    )

clean_df = (
    audit_df.loc[audit_df["included"]]
    .copy()
    .reset_index(drop=True)
)

excluded_df = (
    audit_df.loc[~audit_df["included"]]
    .copy()
    .reset_index(drop=True)
)

duplicates_df = audit_df.loc[
    audit_df["group_size"].gt(1)
].copy()

In [10]:
# ============================================================
# 5. 核验清洗结果，统计各集合、各类别数量
# ============================================================

for column in ["file_sha256", "rgb_sha256"]:
    assert not clean_df[column].duplicated().any(), (
        f"清洗后仍存在 {column} 重复"
    )

assert len(clean_df) + len(excluded_df) == len(audit_df)
assert clean_df["image_error"].eq("").all()

assert not audit_df.loc[
    audit_df["group_size"].gt(1), "included"
].any()

# 补齐所有类别 × 三个集合，避免统计时隐藏零样本类别。
grid = pd.MultiIndex.from_product(
    [class_df["class_id"].tolist(), SPLITS],
    names=["class_id", "split"]
)

counts_df = pd.DataFrame({
    "before": (
        audit_df.groupby(["class_id", "split"])
        .size()
        .reindex(grid, fill_value=0)
    ),
    "after": (
        clean_df.groupby(["class_id", "split"])
        .size()
        .reindex(grid, fill_value=0)
    )
}).reset_index()

counts_df["excluded"] = (
    counts_df["before"] - counts_df["after"]
)

counts_df = counts_df.merge(
    class_df,
    on="class_id",
    how="left"
)

split_counts_df = (
    counts_df.groupby("split")[["before", "after", "excluded"]]
    .sum()
    .reindex(SPLITS)
)

duplicate_groups_df = duplicates_df.drop_duplicates(
    "duplicate_group_id"
)

summary = {
    "policy": "exclude_all_members_of_every_exact_duplicate_group",
    "source_record": "https://zenodo.org/records/17719108",

    # 本 cell 记录本地数据校验信息，不独立验证下载版本。
    "release_verified": False,

    "metadata_sha256": hashlib.sha256(
        metadata_path.read_bytes()
    ).hexdigest(),

    "rgb_hash_definition": (
        "PlantSeg-RGB-v1 NUL prefix + big-endian uint64 width,height "
        "+ row-major RGB bytes; no EXIF transpose"
    ),

    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "pillow": PIL.__version__
    },

    "hosts": sorted(HOSTS),
    "classes": len(class_df),
    "before": len(audit_df),
    "after": len(clean_df),
    "excluded": len(excluded_df),
    "invalid_images": int((~valid).sum()),
    "duplicate_groups": len(duplicate_groups_df),
    "duplicate_records": len(duplicates_df),

    "cross_split_groups": int(
        duplicate_groups_df["cross_split"].sum()
    ),

    "conflicting_label_groups": int(
        duplicate_groups_df["label_conflict"].sum()
    ),

    "near_duplicates_checked": False
}

In [15]:
# ============================================================
# 6. 保存清单、统计、规则摘要与输出文件校验值
# ============================================================

# 只写入审计结果；不修改原始图片。
# 再次运行会更新相同目录下的这些输出文件。
OUT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "audit_manifest.csv": audit_df,
    "clean_manifest.csv": clean_df,
    "excluded_records.csv": excluded_df,
    "duplicate_records.csv": duplicates_df,
    "class_mapping.csv": class_df,
    "class_counts.csv": counts_df,
    "split_counts.csv": split_counts_df.reset_index()
}

for name, frame in tables.items():
    frame.to_csv(
        OUT_DIR / name,
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n"
    )

(OUT_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8"
)

output_names = [*tables, "summary.json"]

checksums = {
    name: hashlib.sha256(
        (OUT_DIR / name).read_bytes()
    ).hexdigest()
    for name in output_names
}

(OUT_DIR / "checksums.json").write_text(
    json.dumps(checksums, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8"
)


# ============================================================
# 7. 展示结果
# ============================================================

display(split_counts_df)

print(
    f"重复组：{summary['duplicate_groups']}；"
    f"涉及记录：{len(duplicates_df)}"
)

print(
    f"跨集合重复组：{summary['cross_split_groups']}；"
    f"标签冲突组：{summary['conflicting_label_groups']}"
)

print(
    f"无法读取：{summary['invalid_images']}；"
    f"最终保留：{len(clean_df):,}"
)

print("各集合清洗后覆盖的类别数：")
print(
    counts_df.loc[counts_df["after"].gt(0)]
    .groupby("split")
    .size()
    .reindex(SPLITS, fill_value=0)
)

zero_counts = counts_df.loc[counts_df["after"].eq(0)]

if not zero_counts.empty:
    print(
        "注意：以下类别在对应集合中无样本；"
        "不自动重新划分或改变清洗规则。"
    )
    display(zero_counts)

print(f"输出目录：{OUT_DIR}")

print(
    "clean_manifest.csv SHA-256："
    f"{checksums['clean_manifest.csv']}"
)


,before,after,excluded
split,,,
train,3007,2780,227
val,475,440,35
test,867,800,67


重复组：162；涉及记录：329
跨集合重复组：73；标签冲突组：34
无法读取：0；最终保留：4,020
各集合清洗后覆盖的类别数：
split
train    43
val      43
test     43
dtype: int64
输出目录：C:\Users\Minghao Lin\AIP_LAB\CW2\artifacts\data_cleaning
clean_manifest.csv SHA-256：0355ecc1d85eee21f06f53ab346de62756ec78e48111eec8ae12916e8c4cd4d6
